# NB03：灰分與鈉離子（Ash Content and Sodium Ion Determination）

**食品分析實驗 週次 3｜Nielsen's Food Analysis, 6th ed. 對應章節：灰分分析（Ash）、礦物質分析（Minerals）**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 由坩堝空重、坩堝+樣品重、灰化後坩堝+灰分重，計算**灰分 %**，並比較黑糖 vs 白糖、糙米 vs 白米。
2. 由 Na⁺ ISE 讀數與**稀釋倍數**回推樣品的鈉濃度，換算成 **NaCl 當量**（× 2.542 = 58.44/22.99）。
3. 計算**標準添加回收率 %**，用 90–110% 的教學判準評估方法準確性。
4. 檢查**飄移(drift)**：每測 10 個樣品重測一次標準液，抓出偏差 > ±5% 的情況。
5. 比較量測鈉 vs 營養標示鈉（換算為每 100 g），並討論非 NaCl 的含鈉來源。
6. （選用）由兩個標準液的 mV 讀數計算 Nernst 響應斜率，跟理論值 59.2 mV/decade 比較。

## 如何使用本筆記本

用法與 NB01/NB02 相同：直接「全部執行」使用內建模擬資料；`load_data(source)` 支援本機路徑、已發布 Google 試算表 CSV 連結、Colab 上傳，或退回內嵌字串。

> ⚠️ **本筆記本內建的資料是模擬教學資料（simulated teaching data）**，不是真實量測值。

## 資料格式 (Data Schema)

沿用共用長格式，欄位同 NB02（含 `sample`）：`group, student, item, rep, value, unit, temp_C, note, sample`。本週的 `item` 值包括：`crucible_empty_g`、`crucible_sample_g`、`crucible_ash_g`（灰分）；`na_ise_ppm`（Na⁺讀數）；`std_addition_before_ppm`/`std_addition_after_ppm`（標準添加回收）；`drift_check_ppm`（飄移監控，`group=0` 代表 QC 標準液，非特定組別）；`nernst_mV`（兩點 Nernst 響應）。

## 1. 環境設定 (Setup)

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded sample data (fallback) ----------------------------------
SAMPLE_CSV = """group,student,item,rep,value,unit,temp_C,note,sample
1,G1_S1,crucible_empty_g,1,28.6437,g,555.0,,black_sugar
1,G1_S1,crucible_sample_g,1,31.0917,g,555.0,,black_sugar
1,G1_S1,crucible_ash_g,1,28.6815,g,555.0,,black_sugar
1,G1_S1,crucible_empty_g,1,24.5651,g,548.0,,white_sugar
1,G1_S1,crucible_sample_g,1,27.0,g,548.0,,white_sugar
1,G1_S1,crucible_ash_g,1,24.5659,g,548.0,,white_sugar
1,G1_S1,crucible_empty_g,1,24.7687,g,554.0,,brown_rice
1,G1_S1,crucible_sample_g,1,27.226,g,554.0,,brown_rice
1,G1_S1,crucible_ash_g,1,24.8,g,554.0,,brown_rice
1,G1_S1,crucible_empty_g,1,27.8632,g,546.0,,polished_rice
1,G1_S1,crucible_sample_g,1,30.4196,g,546.0,,polished_rice
1,G1_S1,crucible_ash_g,1,27.8765,g,546.0,,polished_rice
1,G1_S1,crucible_empty_g,1,27.3275,g,550.0,,milk_powder
1,G1_S1,crucible_sample_g,1,29.7796,g,550.0,,milk_powder
1,G1_S1,crucible_ash_g,1,27.4826,g,550.0,,milk_powder
2,G2_S1,crucible_empty_g,1,28.5485,g,549.0,,black_sugar
2,G2_S1,crucible_sample_g,1,31.0145,g,549.0,,black_sugar
2,G2_S1,crucible_ash_g,1,28.5877,g,549.0,,black_sugar
2,G2_S1,crucible_empty_g,1,28.6703,g,552.0,,white_sugar
2,G2_S1,crucible_sample_g,1,31.1527,g,552.0,,white_sugar
2,G2_S1,crucible_ash_g,1,28.6712,g,552.0,,white_sugar
2,G2_S1,crucible_empty_g,1,24.9257,g,548.0,,brown_rice
2,G2_S1,crucible_sample_g,1,27.4472,g,548.0,,brown_rice
2,G2_S1,crucible_ash_g,1,24.9604,g,548.0,,brown_rice
2,G2_S1,crucible_empty_g,1,25.955,g,556.0,,polished_rice
2,G2_S1,crucible_sample_g,1,28.4143,g,556.0,,polished_rice
2,G2_S1,crucible_ash_g,1,25.9679,g,556.0,,polished_rice
2,G2_S1,crucible_empty_g,1,24.7795,g,553.0,,milk_powder
2,G2_S1,crucible_sample_g,1,27.2375,g,553.0,,milk_powder
2,G2_S1,crucible_ash_g,1,24.9195,g,553.0,,milk_powder
3,G3_S1,crucible_empty_g,1,26.6229,g,551.0,,black_sugar
3,G3_S1,crucible_sample_g,1,29.1501,g,551.0,,black_sugar
3,G3_S1,crucible_ash_g,1,26.6584,g,551.0,,black_sugar
3,G3_S1,crucible_empty_g,1,26.3249,g,551.0,,white_sugar
3,G3_S1,crucible_sample_g,1,28.8358,g,551.0,,white_sugar
3,G3_S1,crucible_ash_g,1,26.3259,g,551.0,,white_sugar
3,G3_S1,crucible_empty_g,1,25.1994,g,553.0,,brown_rice
3,G3_S1,crucible_sample_g,1,27.7028,g,553.0,,brown_rice
3,G3_S1,crucible_ash_g,1,25.23,g,553.0,,brown_rice
3,G3_S1,crucible_empty_g,1,28.231,g,547.0,,polished_rice
3,G3_S1,crucible_sample_g,1,30.715,g,547.0,,polished_rice
3,G3_S1,crucible_ash_g,1,28.243,g,547.0,,polished_rice
3,G3_S1,crucible_empty_g,1,24.8388,g,555.0,,milk_powder
3,G3_S1,crucible_sample_g,1,27.4135,g,555.0,,milk_powder
3,G3_S1,crucible_ash_g,1,24.985,g,555.0,,milk_powder
4,G4_S1,crucible_empty_g,1,27.3914,g,553.0,,black_sugar
4,G4_S1,crucible_sample_g,1,29.8747,g,553.0,,black_sugar
4,G4_S1,crucible_ash_g,1,27.4283,g,553.0,,black_sugar
4,G4_S1,crucible_empty_g,1,27.3552,g,548.0,,white_sugar
4,G4_S1,crucible_sample_g,1,29.8949,g,548.0,,white_sugar
4,G4_S1,crucible_ash_g,1,27.3559,g,548.0,,white_sugar
4,G4_S1,crucible_empty_g,1,25.2875,g,544.0,,brown_rice
4,G4_S1,crucible_sample_g,1,27.7779,g,544.0,,brown_rice
4,G4_S1,crucible_ash_g,1,25.3148,g,544.0,,brown_rice
4,G4_S1,crucible_empty_g,1,24.3498,g,553.0,,polished_rice
4,G4_S1,crucible_sample_g,1,26.8747,g,553.0,,polished_rice
4,G4_S1,crucible_ash_g,1,24.3626,g,553.0,,polished_rice
4,G4_S1,crucible_empty_g,1,27.3422,g,548.0,,milk_powder
4,G4_S1,crucible_sample_g,1,29.8501,g,548.0,,milk_powder
4,G4_S1,crucible_ash_g,1,27.4985,g,548.0,,milk_powder
5,G5_S1,crucible_empty_g,1,28.8841,g,548.0,,black_sugar
5,G5_S1,crucible_sample_g,1,31.351,g,548.0,,black_sugar
5,G5_S1,crucible_ash_g,1,28.9195,g,548.0,,black_sugar
5,G5_S1,crucible_empty_g,1,28.3342,g,550.0,,white_sugar
5,G5_S1,crucible_sample_g,1,30.8585,g,550.0,,white_sugar
5,G5_S1,crucible_ash_g,1,28.3348,g,550.0,,white_sugar
5,G5_S1,crucible_empty_g,1,24.9139,g,550.0,,brown_rice
5,G5_S1,crucible_sample_g,1,27.4362,g,550.0,,brown_rice
5,G5_S1,crucible_ash_g,1,24.9455,g,550.0,,brown_rice
5,G5_S1,crucible_empty_g,1,25.8091,g,543.0,,polished_rice
5,G5_S1,crucible_sample_g,1,28.3051,g,543.0,,polished_rice
5,G5_S1,crucible_ash_g,1,25.8199,g,543.0,,polished_rice
5,G5_S1,crucible_empty_g,1,24.708,g,545.0,,milk_powder
5,G5_S1,crucible_sample_g,1,27.1581,g,545.0,,milk_powder
5,G5_S1,crucible_ash_g,1,24.8586,g,545.0,,milk_powder
1,G1_S1,na_ise_ppm,1,1198.4,ppm,25.9,,soy_sauce
1,G1_S1,na_ise_ppm,1,273.9,ppm,25.4,,potato_chips
1,G1_S1,na_ise_ppm,1,153.2,ppm,24.5,,instant_soup
1,G1_S1,na_ise_ppm,1,200.4,ppm,25.8,,sports_drink
1,G1_S1,na_ise_ppm,1,232.0,ppm,24.4,,milk_powder
2,G2_S1,na_ise_ppm,1,1216.8,ppm,25.2,,soy_sauce
2,G2_S1,na_ise_ppm,1,272.4,ppm,25.7,,potato_chips
2,G2_S1,na_ise_ppm,1,156.2,ppm,25.4,,instant_soup
2,G2_S1,na_ise_ppm,1,205.6,ppm,25.3,,sports_drink
2,G2_S1,na_ise_ppm,1,235.7,ppm,25.3,,milk_powder
3,G3_S1,na_ise_ppm,1,1183.8,ppm,24.8,,soy_sauce
3,G3_S1,na_ise_ppm,1,277.3,ppm,25.0,,potato_chips
3,G3_S1,na_ise_ppm,1,158.5,ppm,24.3,,instant_soup
3,G3_S1,na_ise_ppm,1,211.0,ppm,25.2,,sports_drink
3,G3_S1,na_ise_ppm,1,224.9,ppm,25.9,,milk_powder
4,G4_S1,na_ise_ppm,1,1198.2,ppm,24.7,,soy_sauce
4,G4_S1,na_ise_ppm,1,279.8,ppm,24.0,,potato_chips
4,G4_S1,na_ise_ppm,1,151.6,ppm,25.0,,instant_soup
4,G4_S1,na_ise_ppm,1,206.0,ppm,24.2,,sports_drink
4,G4_S1,na_ise_ppm,1,237.6,ppm,25.0,,milk_powder
5,G5_S1,na_ise_ppm,1,1216.4,ppm,25.1,,soy_sauce
5,G5_S1,na_ise_ppm,1,263.2,ppm,24.5,,potato_chips
5,G5_S1,na_ise_ppm,1,153.6,ppm,25.0,,instant_soup
5,G5_S1,na_ise_ppm,1,201.9,ppm,24.0,,sports_drink
5,G5_S1,na_ise_ppm,1,233.1,ppm,25.8,,milk_powder
1,G1_S2,std_addition_before_ppm,1,297.6,ppm,25.0,,soy_sauce
1,G1_S2,std_addition_after_ppm,1,497.8,ppm,25.0,,soy_sauce
2,G2_S2,std_addition_before_ppm,1,274.9,ppm,25.0,,soy_sauce
2,G2_S2,std_addition_after_ppm,1,477.7,ppm,25.0,,soy_sauce
3,G3_S2,std_addition_before_ppm,1,299.2,ppm,25.0,,soy_sauce
3,G3_S2,std_addition_after_ppm,1,503.2,ppm,25.0,,soy_sauce
4,G4_S2,std_addition_before_ppm,1,302.0,ppm,25.0,,soy_sauce
4,G4_S2,std_addition_after_ppm,1,453.7,ppm,25.0,回收率 < 90%，疑似基質效應或加標體積誤差，建議重做,soy_sauce
5,G5_S2,std_addition_before_ppm,1,292.5,ppm,25.0,,soy_sauce
5,G5_S2,std_addition_after_ppm,1,485.6,ppm,25.0,,soy_sauce
0,QC_standard,drift_check_ppm,0,988.4,ppm,25.0,,Na_1000ppm_std
0,QC_standard,drift_check_ppm,10,991.3,ppm,25.0,,Na_1000ppm_std
0,QC_standard,drift_check_ppm,20,1025.5,ppm,25.0,,Na_1000ppm_std
0,QC_standard,drift_check_ppm,30,990.1,ppm,25.0,,Na_1000ppm_std
0,QC_standard,drift_check_ppm,40,1061.0,ppm,25.0,,Na_1000ppm_std
0,QC_standard,drift_check_ppm,50,989.2,ppm,25.0,,Na_1000ppm_std
1,G1_S3,nernst_mV,1,101.1,mV,25.0,,std_100ppm
1,G1_S3,nernst_mV,1,159.7,mV,25.0,,std_1000ppm
2,G2_S3,nernst_mV,1,99.8,mV,25.0,,std_100ppm
2,G2_S3,nernst_mV,1,159.2,mV,25.0,,std_1000ppm
3,G3_S3,nernst_mV,1,99.2,mV,25.0,,std_100ppm
3,G3_S3,nernst_mV,1,159.7,mV,25.0,,std_1000ppm
4,G4_S3,nernst_mV,1,99.5,mV,25.0,,std_100ppm
4,G4_S3,nernst_mV,1,157.7,mV,25.0,,std_1000ppm
5,G5_S3,nernst_mV,1,98.5,mV,25.0,,std_100ppm
5,G5_S3,nernst_mV,1,159.4,mV,25.0,,std_1000ppm
"""

In [ ]:
def load_data(source=None):
    """Load NB03 ash/sodium data in the shared long-format schema:
    group, student, item, rep, value, unit, temp_C, note, sample
    (see NB01/NB02 for the load_data() source-resolution rules)."""
    if source is not None:
        try:
            return pd.read_csv(source)
        except Exception as e:
            print(f"讀取 {source} 失敗，改用內建模擬資料。錯誤訊息：{e}")

    local_path = os.path.join("data", "nb03_sample.csv")
    if os.path.exists(local_path):
        return pd.read_csv(local_path)

    return pd.read_csv(io.StringIO(SAMPLE_CSV))


df = load_data()
print(f"讀入 {len(df)} 筆資料，共 {df['item'].nunique()} 個量測項目、{df['sample'].nunique()} 種樣品。")
df.head()

### （選用）在 Colab 上傳班級實際資料

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_data(fname)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_data('本機路徑或URL')")

## 2. 灰分 % (Ash Content)

$$灰分\% = \frac{灰分重}{樣品重} \times 100$$

### 範例（沿用課程講義數字，黑糖）

坩堝空重 W₀ = **26.184 g**；坩堝＋樣品 = 28.696 g → 樣品重 = **2.512 g**；灰化後坩堝＋灰分(恆重) = 26.221 g → 灰分重 = **0.037 g**；灰分 % = 0.037 ÷ 2.512 × 100 ≈ **1.47%**。

In [ ]:
W0, W_sample_total, W_ash_total = 26.184, 28.696, 26.221
sample_mass = W_sample_total - W0
ash_mass = W_ash_total - W0
ash_pct_example = ash_mass / sample_mass * 100
print(f"樣品重 = {sample_mass:.3f} g, 灰分重 = {ash_mass:.3f} g, 灰分% = {ash_pct_example:.2f}%  (expect ~1.47%)")

In [ ]:
ash_items = ["crucible_empty_g", "crucible_sample_g", "crucible_ash_g"]
ash_wide = (
    df[df["item"].isin(ash_items)]
    .pivot_table(index=["group", "sample"], columns="item", values="value", aggfunc="first")
    .reset_index()
)
ash_wide["sample_mass_g"] = ash_wide["crucible_sample_g"] - ash_wide["crucible_empty_g"]
ash_wide["ash_mass_g"] = ash_wide["crucible_ash_g"] - ash_wide["crucible_empty_g"]
ash_wide["ash_pct"] = ash_wide["ash_mass_g"] / ash_wide["sample_mass_g"] * 100
ash_wide[["group", "sample", "sample_mass_g", "ash_mass_g", "ash_pct"]]

In [ ]:
ash_summary = ash_wide.groupby("sample")["ash_pct"].agg(n="count", mean_pct="mean", sd_pct="std").reset_index()
ash_summary["rsd_pct"] = ash_summary["sd_pct"] / ash_summary["mean_pct"] * 100
ash_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
order = ash_summary.sort_values("mean_pct")["sample"]
ax.bar(order, ash_summary.set_index("sample").loc[order, "mean_pct"],
       yerr=ash_summary.set_index("sample").loc[order, "sd_pct"], capsize=4, color="tab:brown", alpha=0.85)
ax.set_ylabel("Ash content (%)")
ax.set_title("Ash % by sample (black vs white sugar, brown vs polished rice, blind sample)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

黑糖/糙米的灰分明顯高於白糖/白米——這是**精製程度**的指標之一，但灰分本身**不能**告訴你礦物質的種類或鈉/鹽含量，只能告訴你「總無機殘留物」的比例（延續第 3–4 節，用 Na⁺ ISE 才能回答「鈉」的問題）。全脂奶粉是本課程共同盲樣，此處的灰分結果會延續到後續週次的品管追蹤。

## 3. Na⁺ ISE 讀數、稀釋倍數、NaCl 當量換算

$$C_{樣品} = 讀數 \times 稀釋倍數 \qquad NaCl\ 當量 = Na \times 2.542\ \left(=\frac{58.44}{22.99}\right)$$

### 範例（沿用課程講義數字）

醬油以去離子水稀釋 **1:50** 後，用 Na⁺ ISE 測得讀數 **1100 ppm(mg/L)**：

In [ ]:
NACL_FACTOR = 58.44 / 22.99
reading_ppm, dilution = 1100.0, 50
C_Na_gL = reading_ppm * dilution / 1000.0   # g/L in the original soy sauce
C_Na_100mL = C_Na_gL / 10.0                  # g/100 mL
C_NaCl_100mL = C_Na_100mL * NACL_FACTOR
print(f"Na 濃度 = {C_Na_100mL:.2f} g/100 mL")
print(f"NaCl 當量 = {C_NaCl_100mL:.2f} g/100 mL  (expect ~13.98)")

In [ ]:
na_df = df[df["item"] == "na_ise_ppm"][["group", "sample", "value"]].rename(columns={"value": "reading_ppm"})
na_df

## 4. 標準添加回收率 (Standard-Addition Recovery)

$$回收率\% = \frac{C_{加標} - C_{未加標}}{C_{加標量}} \times 100$$

教學上常以 **90–110%** 作為可接受範圍。

In [ ]:
STD_ADD_PPM = 200.0
before = df[df["item"] == "std_addition_before_ppm"][["group", "value"]].rename(columns={"value": "c_before"})
after = df[df["item"] == "std_addition_after_ppm"][["group", "value", "note"]].rename(columns={"value": "c_after"})
recovery = before.merge(after, on="group")
recovery["recovery_pct"] = (recovery["c_after"] - recovery["c_before"]) / STD_ADD_PPM * 100
recovery["pass_90_110"] = recovery["recovery_pct"].between(90, 110)
recovery

## 5. 飄移檢查 (Drift Check)

每量測約 10 個樣品，重新測一次已知濃度的標準液（此處標準液標稱濃度 1000 ppm），偏差超過 **±5%** 應重新校正並回頭檢視受影響的樣品。

In [ ]:
drift = df[df["item"] == "drift_check_ppm"][["rep", "value"]].rename(columns={"rep": "sample_index", "value": "reading_ppm"})
NOMINAL_STD_PPM = 1000.0
drift["deviation_pct"] = (drift["reading_ppm"] - NOMINAL_STD_PPM) / NOMINAL_STD_PPM * 100
drift["flagged"] = drift["deviation_pct"].abs() > 5
drift

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(drift["sample_index"], drift["deviation_pct"], "o-", color="tab:blue")
ax.axhspan(-5, 5, color="tab:green", alpha=0.12, label="acceptable (+/-5%)")
flagged = drift[drift["flagged"]]
if len(flagged) > 0:
    ax.scatter(flagged["sample_index"], flagged["deviation_pct"], s=120, facecolors="none",
               edgecolors="tab:red", linewidths=2, label="flagged: drift > 5%", zorder=5)
ax.axhline(0, color="grey", linewidth=1)
ax.set_xlabel("Sample index (standard re-measured every 10 samples)")
ax.set_ylabel("Deviation from nominal 1000 ppm (%)")
ax.set_title("Drift check across the measurement run")
ax.legend()
plt.tight_layout()
plt.show()

## 6. 量測鈉 vs 營養標示鈉

把量測結果換算成「每 100 g」跟包裝上「每份」的標示鈉互相比較（先把標示的 mg/份 除以每份克數再乘 100 換算成同一基礎）。差異可能來自：秤重/稀釋誤差、批次差異，以及**非 NaCl 的含鈉來源**（味精 MSG、小蘇打 NaHCO₃、苯甲酸鈉等）——這些都貢獻鈉，但不是鹽。

In [ ]:
NA_PRODUCTS = {
    "soy_sauce":     {"dilution": 50, "label_mg_serving": 850, "serving_g": 15},
    "potato_chips":  {"dilution": 1,  "label_mg_serving": 180, "serving_g": 30},
    "instant_soup":  {"dilution": 20, "label_mg_serving": 750, "serving_g": 250},
    "sports_drink":  {"dilution": 1,  "label_mg_serving": 110, "serving_g": 500},
    "milk_powder":   {"dilution": 20, "label_mg_serving": 120, "serving_g": 25},
}

rows = []
for product, cfg in NA_PRODUCTS.items():
    sub = na_df[na_df["sample"] == product]
    mean_reading = sub["reading_ppm"].mean()
    # potato chips: 5 g sample extracted into 100 mL, measured on the
    # extract directly (dilution factor folded into the 100/5 mL:g ratio)
    if product == "potato_chips":
        measured_mg_100g = mean_reading * 100.0 / 1000.0 * (100.0 / 5.0)
    else:
        measured_mg_100g = mean_reading * cfg["dilution"] / 1000.0 * 100.0
    label_mg_100g = cfg["label_mg_serving"] / cfg["serving_g"] * 100.0
    rows.append([product, label_mg_100g, measured_mg_100g])

compare_df = pd.DataFrame(rows, columns=["product", "label_mg_100g", "measured_mg_100g"])
compare_df["diff_pct"] = (compare_df["measured_mg_100g"] - compare_df["label_mg_100g"]) / compare_df["label_mg_100g"] * 100
compare_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(compare_df))
width = 0.35
ax.bar(x - width / 2, compare_df["label_mg_100g"], width, label="label (mg/100g)", color="tab:blue")
ax.bar(x + width / 2, compare_df["measured_mg_100g"], width, label="measured (mg/100g)", color="tab:orange")
ax.set_xticks(x)
ax.set_xticklabels(compare_df["product"], rotation=20)
ax.set_ylabel("Sodium (mg / 100 g)")
ax.set_title("Measured vs label sodium")
ax.legend()
plt.tight_layout()
plt.show()

## 7. （選用）簡易 Nernst 斜率檢查

$$slope = \frac{mV_2 - mV_1}{\log_{10}C_2 - \log_{10}C_1}$$

理論值於 25°C 約為 **59.2 mV/decade**。若電極健康狀況不佳，實測斜率會明顯偏離理論值。

In [ ]:
nernst = df[df["item"] == "nernst_mV"][["group", "sample", "value"]].rename(columns={"value": "mV"})
nernst["conc_ppm"] = nernst["sample"].str.extract(r"(\d+)").astype(float)
nernst_wide = nernst.pivot_table(index="group", columns="conc_ppm", values="mV").reset_index()
nernst_wide.columns = ["group", "mV_100ppm", "mV_1000ppm"]
nernst_wide["slope_mV_per_decade"] = (
    (nernst_wide["mV_1000ppm"] - nernst_wide["mV_100ppm"])
    / (np.log10(1000.0) - np.log10(100.0))
)
nernst_wide

In [ ]:
mean_slope = nernst_wide["slope_mV_per_decade"].mean()
print(f"班級平均斜率 = {mean_slope:.1f} mV/decade（理論值 59.2 mV/decade，25°C）")

## 8. 練習題 (Exercises)

### 問題 1：標準添加回收率判斷

某組加標前讀數 305 ppm、加標後讀數 480 ppm、加標量 200 ppm。計算回收率%，並判斷是否落在 90–110% 的可接受範圍。若不合格，可能的原因有哪些？

In [ ]:
# TODO: 請在這裡作答

### 問題 2：NaCl 當量換算

某即食湯粉以去離子水稀釋 1:20 後，Na⁺ ISE 讀數為 **420 ppm**。求原湯粉萃取液中的 Na 濃度(g/100 mL)與 NaCl 當量(g/100 mL)。

In [ ]:
# TODO: 請在這裡作答

### 問題 3：灰分能回答什麼

用第 2 節的黑糖 vs 白糖灰分數據，說明「灰分%的差異」能不能證明黑糖含有更多對健康有益的礦物質？灰分這個指標的極限在哪裡？

In [ ]:
# TODO: 請在這裡作答

### 問題 4：標示 vs 量測的落差

用第 6 節 `compare_df` 找出「量測值低於標示值」的產品，並討論可能原因（提示：不是每個差異都代表標示造假）。

In [ ]:
# TODO: 請在這裡作答

### 問題 5：非 NaCl 的鈉來源

如果一款產品標示鈉 400 mg/100g，但配方中同時含有食鹽、味精(MSG)、小蘇打(NaHCO₃)，直接把「鈉 × 2.542」當作「鹽含量」合理嗎？為什麼？

In [ ]:
# TODO: 請在這裡作答

## 9. 匯出結果 (Export)

In [ ]:
output_path = "nb03_ash_na_results.csv"
ash_summary.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")